# Superstore - Retail

In [1]:
import pandas as pd 
import numpy as np 

In [2]:
data = pd.read_csv("../data/raw/superstore-tableau.csv")

In [3]:
data.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,08-11-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,12-06-2016,16-06-2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,11-10-2015,18-10-2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [4]:
data.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='str')

In [5]:
# Creating unique data tables

customer = data[["Customer ID", "Customer Name", "Segment", "Country", "State", "City", "Postal Code", "Region"]].drop_duplicates()
product = data[["Product ID", "Category", "Sub-Category", "Product Name"]].drop_duplicates()
orders = data[["Customer ID", "Order ID", "Order Date", "Ship Date", "Ship Mode", "Product ID", "Sales", "Quantity", "Discount", "Profit"]].drop_duplicates()
orders['Order Date'] = pd.to_datetime(orders['Order Date'], format = "mixed")
orders['Ship Date'] = pd.to_datetime(orders['Ship Date'], format = "mixed")
orders["days_till_shipped"] = (orders["Order Date"] - orders["Ship Date"]).dt.days

dates = pd.DataFrame(orders['Order Date'].unique()).sort_values(by=0).reset_index(drop=True)
dates.columns = ['date']
dates['month'] = dates['date'].dt.month_name()
dates['month_num'] = dates['date'].dt.month
dates['year'] = dates['date'].dt.year



In [6]:
# Let's check total  orders
print("Total orders:", len(orders['Order ID'].unique()))

# Let's check total  sales
print("Total sales:", round(orders['Sales'].sum(), 2))

# Let's check total  customers
print("Total customers:", len(orders['Customer ID'].unique()))


Total orders: 5009
Total sales: 2296919.49
Total customers: 793


In [7]:
# Total Sales Year-on-Year
sales_yoy = pd.DataFrame(orders.merge(dates[['date','year']], left_on="Order Date", right_on = "date").groupby(['year'])['Sales'].sum())
x1 = sales_yoy.Sales.values[0]
cum_sales = []
for x in sales_yoy.Sales.values:
    if(x==x1):
        cum_sales.append(0)
    else:
        cum_sales.append(round((x-x1)/x,2))
        x1 = x
sales_yoy['%_change_in_sales'] = cum_sales
sales_yoy

,Sales,%_change_in_sales
year,,
2014,483966.1261,0.00
2015,470532.5090,-0.03
2016,609205.5980,0.23
2017,733215.2552,0.17


In [42]:
# Monthly sales across years 2014-2017
sales_2014 = round(pd.DataFrame(orders.merge(dates[dates['year'] == 2014][['date','month_num']], 
                                      left_on="Order Date", 
                                     right_on = "date").groupby(['month_num'])['Sales'].sum()).sort_values('month_num'),2)
sales_2014.columns = ["Sales_2014"]

sales_2015 = round(pd.DataFrame(orders.merge(dates[dates['year'] == 2015][['date','month_num']], 
                                      left_on="Order Date", 
                                     right_on = "date").groupby(['month_num'])['Sales'].sum()).sort_values('month_num'),2)
sales_2015.columns = ["Sales_2015"]

sales_2016 = round(pd.DataFrame(orders.merge(dates[dates['year'] == 2016][['date','month_num']], 
                                      left_on="Order Date", 
                                     right_on = "date").groupby(['month_num'])['Sales'].sum()).sort_values('month_num'),2)
sales_2016.columns = ["Sales_2016"]
sales_2017 = round(pd.DataFrame(orders.merge(dates[dates['year'] == 2017][['date','month_num']], 
                                      left_on="Order Date", 
                                     right_on = "date").groupby(['month_num'])['Sales'].sum()).sort_values('month_num'),2)
sales_2017.columns = ["Sales_2017"]
monthly_sales_all = sales_2014.join([sales_2015, sales_2016, sales_2017])
monthly_sales_all = monthly_sales_all.merge(dates[['month_num',"month"]].drop_duplicates(), "left", on ="month_num")[["month","Sales_2014","Sales_2015","Sales_2016","Sales_2017"]]
monthly_sales_all

,month,Sales_2014,Sales_2015,Sales_2016,Sales_2017
0,January,28953.71,29347.39,38048.18,64734.31
1,February,12743.11,20728.35,49238.41,50011.49
2,March,54801.91,40876.61,49612.04,74774.08
3,April,24428.64,38056.97,45192.28,39072.00
4,May,29639.83,30933.71,64964.32,40882.45
5,June,29287.03,28862.20,38991.94,47742.33
6,July,35341.25,28730.38,42773.40,54382.09
7,August,37854.55,50094.53,46339.99,75675.30
8,September,66110.22,66729.33,41985.14,74164.61
9,October,34561.95,32025.08,52268.15,65501.16


In [46]:
# % change in monthly sales across years 2015-2017
# % of change in sales
monthly_sales_all['%_change_from_2014'] = round((monthly_sales_all['Sales_2015'] - monthly_sales_all['Sales_2014'])/
                                               monthly_sales_all['Sales_2015'],2)
monthly_sales_all['%_change_from_2015'] = round((monthly_sales_all['Sales_2016'] - monthly_sales_all['Sales_2015'])/
                                               monthly_sales_all['Sales_2016'],2)
monthly_sales_all['%_change_from_2016'] = round((monthly_sales_all['Sales_2017'] - monthly_sales_all['Sales_2016'])/
                                               monthly_sales_all['Sales_2017'],2)
monthly_sales_all = monthly_sales_all[["month","Sales_2014","Sales_2015","%_change_from_2014","Sales_2016","%_change_from_2015","Sales_2017","%_change_from_2016"]]
monthly_sales_all

,month,Sales_2014,Sales_2015,%_change_from_2014,Sales_2016,%_change_from_2015,Sales_2017,%_change_from_2016
0,January,28953.71,29347.39,0.01,38048.18,0.23,64734.31,0.41
1,February,12743.11,20728.35,0.39,49238.41,0.58,50011.49,0.02
2,March,54801.91,40876.61,-0.34,49612.04,0.18,74774.08,0.34
3,April,24428.64,38056.97,0.36,45192.28,0.16,39072.00,-0.16
4,May,29639.83,30933.71,0.04,64964.32,0.52,40882.45,-0.59
5,June,29287.03,28862.20,-0.01,38991.94,0.26,47742.33,0.18
6,July,35341.25,28730.38,-0.23,42773.40,0.33,54382.09,0.21
7,August,37854.55,50094.53,0.24,46339.99,-0.08,75675.30,0.39
8,September,66110.22,66729.33,0.01,41985.14,-0.59,74164.61,0.43
9,October,34561.95,32025.08,-0.08,52268.15,0.39,65501.16,0.20


In [52]:
round(monthly_sales_all["%_change_from_2014"].mean(),4), monthly_sales_all["%_change_from_2015"].mean(), round(monthly_sales_all["%_change_from_2016"].mean(),4)

(np.float64(-0.0092), np.float64(0.2075), np.float64(0.1167))

In [53]:
round(monthly_sales_all["Sales_2014"].sum(),4), round(monthly_sales_all["Sales_2015"].sum(),4), round(monthly_sales_all["Sales_2016"].sum(),4), round(monthly_sales_all["Sales_2017"].sum(),4)

(np.float64(483966.13),
 np.float64(470532.52),
 np.float64(609205.6),
 np.float64(733215.26))

The year 2016 observed a 25% of increase in sales in comparison to the previous year. However, 2017 witnessed only 20% of increase in sales in comparison to the previous year, indicating a 5% of drop in sales if consistency was witnessed

In [11]:
orders.head()

,Customer ID,Order ID,Order Date,Ship Date,Ship Mode,Product ID,Sales,Quantity,Discount,Profit,days_till_shipped
0,CG-12520,CA-2016-152156,2016-08-11,2016-11-11,Second Class,FUR-BO-10001798,261.9600,2,0.00,41.9136,-92
1,CG-12520,CA-2016-152156,2016-08-11,2016-11-11,Second Class,FUR-CH-10000454,731.9400,3,0.00,219.5820,-92
2,DV-13045,CA-2016-138688,2016-12-06,2016-06-16,Second Class,OFF-LA-10000240,14.6200,2,0.00,6.8714,173
3,SO-20335,US-2015-108966,2015-11-10,2015-10-18,Standard Class,FUR-TA-10000577,957.5775,5,0.45,-383.0310,23
4,SO-20335,US-2015-108966,2015-11-10,2015-10-18,Standard Class,OFF-ST-10000760,22.3680,2,0.20,2.5164,23


In [59]:
# creating aggregated view of orders
columns = ['Order ID', 'Order Date', 'year', 'month', "month_num", 'Ship Mode', 'days_till_shipped', 'Sales',
       'Quantity', 'Product ID' ]
orders_agg = orders.groupby(["Order ID", 
                             "Order Date", 
                             "Ship Mode",
                             "days_till_shipped"])[
                                 ["Sales",
                                  "Quantity"]
                                 ].sum().reset_index().merge(
                                     orders.groupby("Order ID")["Product ID"].count(),
                                      on="Order ID").merge(dates[["date",
                                                        "year","month","month_num"]], 
                                                        left_on="Order Date", 
                                                        right_on = "date")[columns]
orders_agg

,Order ID,Order Date,year,month,month_num,Ship Mode,days_till_shipped,Sales,Quantity,Product ID
0,CA-2014-100006,2014-07-09,2014,July,7,Standard Class,-66,377.970,3,1
1,CA-2014-100090,2014-08-07,2014,August,8,Standard Class,-122,699.192,9,2
2,CA-2014-100293,2014-03-14,2014,March,3,Standard Class,-4,91.056,6,1
3,CA-2014-100328,2014-01-28,2014,January,1,Standard Class,-33,3.928,1,1
4,CA-2014-100363,2014-08-04,2014,August,8,Standard Class,111,21.376,5,2
...,...,...,...,...,...,...,...,...,...,...
5004,US-2017-168802,2017-03-11,2017,March,3,Standard Class,-122,18.368,4,1
5005,US-2017-169320,2017-07-23,2017,July,7,Second Class,-2,171.430,7,2
5006,US-2017-169488,2017-07-09,2017,July,7,First Class,-62,56.860,7,2
5007,US-2017-169502,2017-08-28,2017,August,8,Standard Class,231,113.410,8,2


In [73]:
orders_agg.groupby(["Ship Mode","year"])["Order ID"].count().reset_index()

,Ship Mode,year,Order ID
0,First Class,2014,145
1,First Class,2015,143
2,First Class,2016,215
3,First Class,2017,284
4,Same Day,2014,48
5,Same Day,2015,53
6,Same Day,2016,74
7,Same Day,2017,89
8,Second Class,2014,190
9,Second Class,2015,206


In [ ]:
# Total orders shipped for each ship mode through 2014-2017
orders_agg.groupby(["Ship Mode","year"])["Order ID"].count().reset_index().pivot(
    index = "Ship Mode",
    columns=["year"], 
    values = "Order ID")

year,2014,2015,2016,2017
Ship Mode,,,,
First Class,145,143,215,284
Same Day,48,53,74,89
Second Class,190,206,244,324
Standard Class,586,636,782,990
